In [1]:
# Notebooks live in Notebooks/, but every data path is relative to the repo root.
import os, sys
while not os.path.isdir("data") and os.getcwd() != "/":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: /home/user1/WhatDoLLMsWant


# Books domain — design (2026-10-12)

A second feature domain next to the laptops. Items are **real, famous books**; the prompt shows only
**title, author and year** (no tags), so the model chooses from what it knows about the book. The
**features are hidden**: the genre shelves Goodreads readers put the book on. They are used only in
the analysis (feature BT on the tags) and in contracts ("I prefer fantasy books.").

Source: [goodbooks-10k](https://github.com/zygmuntz/goodbooks-10k) (10,000 most-rated books on
Goodreads, with per-book shelf counts), downloaded to `external/goodbooks-10k/` (gitignored).

Decisions (Lotan, 2026-10-12): real books, as famous as possible; genre tags only for now;
prompt = title + author + year.

Output: `figs/books/books_selected.csv` — the list that goes into `src/alternatives.py`.

In [2]:
import pandas as pd, numpy as np
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 60)
D = "external/goodbooks-10k"
books = pd.read_csv(f"{D}/books.csv")
bt = pd.read_csv(f"{D}/book_tags.csv").merge(pd.read_csv(f"{D}/tags.csv"), on="tag_id")
print(len(books), "books,", bt.tag_name.nunique(), "distinct shelves")

10000 books, 34252 distinct shelves


## 1. From shelves to genre tags

Shelves are user-made, so most of the volume is bookkeeping ("to-read", "owned", "favorites") and the
same genre has many spellings ("sci-fi", "scifi", "science-fiction"). We keep 8 genres and map the
common spellings onto them. `children` and `horror` were dropped: too rare among the famous books
(3 each in the top 40).

In [3]:
GENRES = {
    "fantasy":            ["fantasy", "urban-fantasy", "magic", "high-fantasy", "epic-fantasy"],
    "science-fiction":    ["science-fiction", "sci-fi", "scifi", "dystopian", "dystopia"],
    "romance":            ["romance", "paranormal-romance", "contemporary-romance", "chick-lit", "love"],
    "mystery-thriller":   ["mystery", "thriller", "crime", "suspense", "mystery-thriller", "detective"],
    "young-adult":        ["young-adult", "ya", "teen", "ya-fiction", "young-adult-fiction"],
    "classics":           ["classics", "classic", "literature", "classic-literature"],
    "historical-fiction": ["historical-fiction", "historical", "historical-fiction-romance"],
    "nonfiction":         ["non-fiction", "nonfiction", "memoir", "biography", "history", "science",
                           "philosophy", "self-help", "psychology", "business"],
}
genres = list(GENRES)
syn2g = {s: g for g, ss in GENRES.items() for s in ss}
bt["genre"] = bt.tag_name.map(syn2g)
g = bt.dropna(subset=["genre"]).groupby(["goodreads_book_id", "genre"])["count"].sum().unstack(fill_value=0)
share = g.div(g.sum(axis=1), axis=0)   # a book's genre shelvings, as shares that sum to 1
share.head()

genre,classics,fantasy,historical-fiction,mystery-thriller,nonfiction,romance,science-fiction,young-adult
goodreads_book_id,,,,,,,,
1,0.015600,0.669169,0.0,0.007572,0.0,0.005439,0.000000,0.302221
2,0.029701,0.522337,0.0,0.054860,0.0,0.026141,0.024423,0.342538
3,0.030331,0.677466,0.0,0.008487,0.0,0.000000,0.000000,0.283716
5,0.017107,0.683198,0.0,0.008255,0.0,0.000000,0.000000,0.291440
6,0.021655,0.654291,0.0,0.008282,0.0,0.000000,0.000000,0.315772


A book **has** a genre if that genre is at least 12% of its genre shelvings. The threshold sets
how many tags a book gets; 0.12 gives mostly 1–3 tags, which is what a reader would say.

In [4]:
for thr in [0.08, 0.12, 0.20]:
    n = (share >= thr).sum(axis=1)
    print(f"thr={thr}: tags per book (top 500 by ratings) ->",
          n.reindex(books.sort_values("ratings_count", ascending=False).goodreads_book_id.head(500)).value_counts().sort_index().to_dict())
THR = 0.12
has = (share >= THR).astype(int)

thr=0.08: tags per book (top 500 by ratings) -> {1: 141, 2: 194, 3: 128, 4: 35, 5: 2}
thr=0.12: tags per book (top 500 by ratings) -> {1: 188, 2: 194, 3: 110, 4: 8}
thr=0.2: tags per book (top 500 by ratings) -> {1: 259, 2: 201, 3: 40}


In [5]:
b = books.set_index("goodreads_book_id")
b["title_clean"] = b.title.str.replace(r"\s*\(.*\)\s*$", "", regex=True).str.strip()   # drop "(Series, #1)"
b["author"] = b.authors.str.split(",").str[0].str.strip()                                # first author only
b["year"] = b.original_publication_year
b["series"] = b.title.str.extract(r"\(([^,#]+)[,#]")[0].str.strip()
b["series_no"] = b.title.str.extract(r"#(\d+)")[0].astype(float)
b = b.join(has, how="inner")
b = b[b.language_code.fillna("eng").str.startswith("en") & b.year.notna()]
b["n_tags"] = b[genres].sum(axis=1)
b = b[b.n_tags > 0].sort_values("ratings_count", ascending=False)
print("share of the 500 most-rated books carrying each tag:")
print(b.head(500)[genres].mean().round(2).to_string())

share of the 500 most-rated books carrying each tag:
fantasy               0.32
science-fiction       0.13
romance               0.24
mystery-thriller      0.15
young-adult           0.36
classics              0.39
historical-fiction    0.14
nonfiction            0.14


## 2. Picking the items

Fame first, then balance. Three passes, always one book per series (its first book) and at most two
per author:

1. the **32 most-rated** books, no condition on tags;
2. every tag up to **8 books**, taking the most-rated books that carry it (this is what brings in
   nonfiction and historical fiction);
3. fill up to **48** by rating count, with a cap of 18 per tag (or, if pass 2 overshot, drop the
   last-added books that no tag needs).

Why 48: 48 × 47 = 2,256 ordered pairs × 5 templates = 11,280 rows per run, a bit more than the
45 laptops (9,900). Why a minimum of 8 per tag: a tag weight fitted on fewer books is noise.

In [6]:
N, TOP, MIN, CAP = 48, 32, 8, 18
pick, cnt, series_seen, author_cnt = [], {t: 0 for t in genres}, set(), {}

def ok(gid, r, need=None, cap=None):
    tl = [t for t in genres if r[t]]
    if gid in pick: return False
    if need is not None and need not in tl: return False
    if cap is not None and any(cnt[t] >= cap for t in tl): return False
    if pd.notna(r.series) and (r.series in series_seen or r.series_no > 1): return False
    if author_cnt.get(r.author, 0) >= 2: return False
    return True

def take(gid, r):
    pick.append(gid); series_seen.add(r.series); author_cnt[r.author] = author_cnt.get(r.author, 0) + 1
    for t in genres:
        if r[t]: cnt[t] += 1

for gid, r in b.iterrows():                       # pass 1
    if len(pick) >= TOP: break
    if ok(gid, r): take(gid, r)
for t in genres:                                  # pass 2
    for gid, r in b.iterrows():
        if cnt[t] >= MIN: break
        if ok(gid, r, need=t): take(gid, r)
for gid, r in b.iterrows():                       # pass 3
    if len(pick) >= N: break
    if ok(gid, r, cap=CAP): take(gid, r)
while len(pick) > N:                              # pass 2 can overshoot: drop the last-added book whose tags stay >= MIN without it
    for gid in reversed(pick):
        tl = [t for t in genres if b.loc[gid, t]]
        if all(cnt[t] > MIN for t in tl):
            pick.remove(gid)
            for t in tl: cnt[t] -= 1
            break

sel = b.loc[pick].sort_values("ratings_count", ascending=False).copy()
sel["year"] = sel.year.astype(int)
sel["tags"] = sel[genres].apply(lambda r: ", ".join(t for t in genres if r[t]), axis=1)
sel["pop_rank"] = sel.ratings_count.rank(ascending=False).astype(int)
sel["pop_rank_all"] = [int((books.ratings_count > rc).sum()) + 1 for rc in sel.ratings_count]
sel[["title_clean", "author", "year", "average_rating", "ratings_count", "pop_rank_all", "tags"]]

,title_clean,author,year,average_rating,ratings_count,pop_rank_all,tags
goodreads_book_id,,,,,,,
2767052,The Hunger Games,Suzanne Collins,2008,4.34,4780653,1,"science-fiction, young-adult"
3,Harry Potter and the Sorcerer's Stone,J.K. Rowling,1997,4.44,4602479,2,"fantasy, young-adult"
41865,Twilight,Stephenie Meyer,2005,3.57,3866839,3,"fantasy, young-adult"
2657,To Kill a Mockingbird,Harper Lee,1960,4.25,3198671,4,classics
4671,The Great Gatsby,F. Scott Fitzgerald,1925,3.89,2683664,5,classics
11870085,The Fault in Our Stars,John Green,2012,4.26,2346404,6,"romance, young-adult"
5907,The Hobbit,J.R.R. Tolkien,1937,4.25,2071616,7,"fantasy, classics"
5107,The Catcher in the Rye,J.D. Salinger,1951,3.79,2044241,8,"young-adult, classics"
1885,Pride and Prejudice,Jane Austen,1813,4.24,2035490,9,"romance, classics"


## 3. Checks: tag coverage, tag overlap, fame

In [7]:
print("books per tag:", cnt)
print("least famous pick is #%d of 10,000 by ratings count" % sel.pop_rank_all.max())
print("tags per book:", sel.n_tags.value_counts().sort_index().to_dict())
sel[genres].corr().round(2)

books per tag: {'fantasy': 10, 'science-fiction': 8, 'romance': 8, 'mystery-thriller': 8, 'young-adult': 13, 'classics': 26, 'historical-fiction': 8, 'nonfiction': 8}
least famous pick is #119 of 10,000 by ratings count
tags per book: {1: 16, 2: 23, 3: 9}


,fantasy,science-fiction,romance,mystery-thriller,young-adult,classics,historical-fiction,nonfiction
fantasy,1.00,-0.09,-0.23,0.05,0.26,-0.04,-0.23,-0.09
science-fiction,-0.09,1.00,-0.20,-0.20,0.10,0.07,-0.20,-0.20
romance,-0.23,-0.20,1.00,-0.20,-0.15,-0.04,0.25,-0.05
mystery-thriller,0.05,-0.20,-0.20,1.00,-0.02,-0.26,-0.20,-0.20
young-adult,0.26,0.10,-0.15,-0.02,1.00,-0.19,-0.02,-0.27
classics,-0.04,0.07,-0.04,-0.26,-0.19,1.00,0.07,-0.15
historical-fiction,-0.23,-0.20,0.25,-0.20,-0.02,0.07,1.00,-0.20
nonfiction,-0.09,-0.20,-0.05,-0.20,-0.27,-0.15,-0.20,1.00


Correlations stay under 0.4 in absolute value (fantasy–young-adult is the largest, +0.31), so the
8 tag weights are separately identifiable. `classics` is on 26 of the 48 books: famous books are old
books, there is no way around it with "as famous as possible".

## 4. What the prompt will show

Only the title, the author and the year. Example of the item string and of the hidden feature
columns that go into `scores.csv`.

In [8]:
def item_text(r):
    return f"{r.title_clean} by {r.author} ({r.year})"
for _, r in sel.head(5).iterrows():
    print(item_text(r), "   <- hidden tags:", r.tags)

The Hunger Games by Suzanne Collins (2008)    <- hidden tags: science-fiction, young-adult
Harry Potter and the Sorcerer's Stone by J.K. Rowling (1997)    <- hidden tags: fantasy, young-adult
Twilight by Stephenie Meyer (2005)    <- hidden tags: fantasy, young-adult
To Kill a Mockingbird by Harper Lee (1960)    <- hidden tags: classics
The Great Gatsby by F. Scott Fitzgerald (1925)    <- hidden tags: classics


In [9]:
out = sel[["title_clean", "author", "year", "average_rating", "ratings_count", "pop_rank_all"] + genres].rename(columns={"title_clean": "title"})
out.index.name = "goodreads_book_id"
out.to_csv("figs/books/books_selected.csv")
print("wrote figs/books/books_selected.csv:", out.shape)

wrote figs/books/books_selected.csv: (48, 14)
